# Sanity Check - Step 01: Split Players

Überprüft:
- Spieler korrekt aufgeteilt
- Kanäle pro Person korrekt zugeordnet
- Kanal-Typen gesetzt
- Status-Kanal vorhanden für beide

In [1]:
import sys
from pathlib import Path
import mne

sys.path.append(str(Path.cwd().parent / 'eeg_pipeline'))
import config

print("Setup erfolgreich")

Setup erfolgreich


## 1. Split Files laden

In [2]:
subject_id = config.SUBJECTS[0]

files = {}
for person in ["P1", "P2"]:
    path = config.OUTPUT_DIR / f"sub-{subject_id}_{person}_split.fif"
    if path.exists():
        files[person] = mne.io.read_raw_fif(str(path), preload=False)
        print(f"✓ {person}: {path.name}")
    else:
        print(f"✗ {person}: File not found")

Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_split.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_30924\3455910378.py:7: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P1_split.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  files[person] = mne.io.read_raw_fif(str(path), preload=False)


Isotrak not found
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P1: sub-01_P1_split.fif
Opening raw data file c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_split.fif...


C:\Users\BKALYON\AppData\Local\Temp\ipykernel_30924\3455910378.py:7: RuntimeWarning: This filename (c:\Users\BKALYON\Bala_Sharks\eeg-bala-sharks\data\sub-01_P2_split.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  files[person] = mne.io.read_raw_fif(str(path), preload=False)


Isotrak not found
    Range : 0 ... 733799 =      0.000 ...  3668.995 secs
Ready.
✓ P2: sub-01_P2_split.fif


## 2. Kanal-Übersicht

In [3]:
for person, raw in files.items():
    print(f"\n=== {person} ===")
    print(f"Gesamt-Kanäle: {len(raw.ch_names)}")
    
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    eog_picks = mne.pick_types(raw.info, eog=True, exclude=[])
    resp_picks = mne.pick_types(raw.info, resp=True, exclude=[])
    misc_picks = mne.pick_types(raw.info, misc=True, exclude=[])
    stim_picks = mne.pick_types(raw.info, stim=True, exclude=[])
    
    print(f"  - EEG: {len(eeg_picks)}")
    print(f"  - EOG: {len(eog_picks)}")
    print(f"  - Respiration: {len(resp_picks)}")
    print(f"  - Misc: {len(misc_picks)}")
    print(f"  - Stim: {len(stim_picks)}")
    
    if len(stim_picks) == 0:
        print(f"  ⚠ Warnung: Kein Stim-Kanal vorhanden")
    
    # Check for old prefixes
    prefix = config.PLAYER_PREFIX_MAP[person]
    old_prefix_count = sum(1 for ch in raw.ch_names if ch.startswith(prefix))
    if old_prefix_count > 0:
        print(f"  ⚠ {old_prefix_count} Kanäle haben noch Prefix '{prefix}'")
    else:
        print(f"  ✓ Prefix '{prefix}' erfolgreich entfernt")


=== P1 ===
Gesamt-Kanäle: 72
  - EEG: 64
  - EOG: 2
  - Respiration: 1
  - Misc: 1
  - Stim: 1
  ⚠ 71 Kanäle haben noch Prefix '2-'

=== P2 ===
Gesamt-Kanäle: 72
  - EEG: 64
  - EOG: 2
  - Respiration: 1
  - Misc: 1
  - Stim: 1
  ⚠ 71 Kanäle haben noch Prefix '1-'


## 3. Kanal-Namen (erste 10)

In [4]:
for person, raw in files.items():
    print(f"\n{person} - Erste 10 Kanal-Namen:")
    for i, ch in enumerate(raw.ch_names[:10], 1):
        ch_type = raw.get_channel_types([ch])[0]
        print(f"  {i:2d}. {ch:10s} ({ch_type})")


P1 - Erste 10 Kanal-Namen:
   1. 2-A1       (eeg)
   2. 2-A2       (eeg)
   3. 2-A3       (eeg)
   4. 2-A4       (eeg)
   5. 2-A5       (eeg)
   6. 2-A6       (eeg)
   7. 2-A7       (eeg)
   8. 2-A8       (eeg)
   9. 2-A9       (eeg)
  10. 2-A10      (eeg)

P2 - Erste 10 Kanal-Namen:
   1. 1-A1       (eeg)
   2. 1-A2       (eeg)
   3. 1-A3       (eeg)
   4. 1-A4       (eeg)
   5. 1-A5       (eeg)
   6. 1-A6       (eeg)
   7. 1-A7       (eeg)
   8. 1-A8       (eeg)
   9. 1-A9       (eeg)
  10. 1-A10      (eeg)
